# E-Commerce Customer Intelligence & Churn Prediction

## XGBoost for Repeat Purchase Prediction

This notebook trains and evaluates an XGBoost model for predicting whether a customer will make another purchase within the defined future observation window.

### Modeling Strategy

- Time-based train/test split
- Historical customer behavior features
- SMOTE for training-class imbalance
- SMOTE applied only inside the training pipeline
- Lightweight RandomizedSearchCV
- Untouched latest test snapshot
- Precision, Recall, F1, ROC-AUC and PR-AUC evaluation
- Threshold analysis
- Feature importance

The latest historical test snapshot is never used during model selection or hyperparameter tuning.

In [2]:
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [3]:
PROJECT_ROOT = Path.cwd().parent

TRAIN_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "repeat_purchase_train.csv"
)

TEST_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "repeat_purchase_test.csv"
)

MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project Root:")
print(PROJECT_ROOT)

print("\nTrain file:")
print(TRAIN_FILE)

print("\nTest file:")
print(TEST_FILE)

Project Root:
d:\Data scientist\E-Commerce Customer Intelligence & Churn Prediction

Train file:
d:\Data scientist\E-Commerce Customer Intelligence & Churn Prediction\data\processed\repeat_purchase_train.csv

Test file:
d:\Data scientist\E-Commerce Customer Intelligence & Churn Prediction\data\processed\repeat_purchase_test.csv


In [4]:
train_df = pd.read_csv(
    TRAIN_FILE
)

test_df = pd.read_csv(
    TEST_FILE
)

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)

Training shape: (115063, 18)
Test shape: (55907, 18)


In [5]:
print("Training snapshots:")

print(
    pd.to_datetime(
        train_df["snapshot_date"]
    ).dt.date.unique()
)

print("\nTest snapshot:")

print(
    pd.to_datetime(
        test_df["snapshot_date"]
    ).dt.date.unique()
)

Training snapshots:
[datetime.date(2017, 5, 1) datetime.date(2017, 6, 30)
 datetime.date(2017, 8, 29) datetime.date(2017, 11, 2)
 datetime.date(2018, 1, 1)]

Test snapshot:
[datetime.date(2018, 3, 2)]


In [6]:
TARGET = "repeat_purchase"

print("Training target:")
print(
    train_df[TARGET]
    .value_counts()
    .sort_index()
)

print("\nTest target:")
print(
    test_df[TARGET]
    .value_counts()
    .sort_index()
)

print("\nTraining repeat rate:")
print(
    f"{train_df[TARGET].mean() * 100:.2f}%"
)

print("\nTest repeat rate:")
print(
    f"{test_df[TARGET].mean() * 100:.2f}%"
)

Training target:
repeat_purchase
0    113393
1      1670
Name: count, dtype: int64

Test target:
repeat_purchase
0    55252
1      655
Name: count, dtype: int64

Training repeat rate:
1.45%

Test repeat rate:
1.17%


In [7]:
FORBIDDEN_COLUMNS = [
    "customer_unique_id",
    "snapshot_date",
    "future_end_date",
    "first_purchase_date",
    "last_purchase_date",
    "future_purchase_count",
    "churn_label",
    "repeat_purchase"
]

print("Forbidden columns:")
print()

for column in FORBIDDEN_COLUMNS:
    print("-", column)

Forbidden columns:

- customer_unique_id
- snapshot_date
- future_end_date
- first_purchase_date
- last_purchase_date
- future_purchase_count
- churn_label
- repeat_purchase


In [8]:
FEATURE_COLUMNS = [
    "total_orders",
    "total_revenue",
    "average_order_value",
    "recency_days",
    "customer_lifetime_days",
    "repeat_customer",
    "purchase_frequency",
    "mean_purchase_gap_days",
    "median_purchase_gap_days",
    "max_purchase_gap_days",
    "purchase_gap_count",
    "observation_window_days"
]

print("XGBoost Features:")
print()

for feature in FEATURE_COLUMNS:
    print("-", feature)

XGBoost Features:

- total_orders
- total_revenue
- average_order_value
- recency_days
- customer_lifetime_days
- repeat_customer
- purchase_frequency
- mean_purchase_gap_days
- median_purchase_gap_days
- max_purchase_gap_days
- purchase_gap_count
- observation_window_days


In [9]:
missing_train = [
    feature
    for feature in FEATURE_COLUMNS
    if feature not in train_df.columns
]

missing_test = [
    feature
    for feature in FEATURE_COLUMNS
    if feature not in test_df.columns
]

print("Missing train features:")
print(missing_train)

print("\nMissing test features:")
print(missing_test)

Missing train features:
[]

Missing test features:
[]


In [10]:
X_train = train_df[
    FEATURE_COLUMNS
].copy()

y_train = train_df[
    TARGET
].astype(int)

X_test = test_df[
    FEATURE_COLUMNS
].copy()

y_test = test_df[
    TARGET
].astype(int)

X_train = X_train.apply(
    pd.to_numeric,
    errors="coerce"
)

X_test = X_test.apply(
    pd.to_numeric,
    errors="coerce"
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (115063, 12)
y_train: (115063,)
X_test: (55907, 12)
y_test: (55907,)


In [11]:
print("Training missing values:")
display(
    X_train.isna().sum()
)

print("\nTest missing values:")
display(
    X_test.isna().sum()
)

Training missing values:


total_orders                     0
total_revenue                    0
average_order_value              0
recency_days                     0
customer_lifetime_days           0
repeat_customer                  0
purchase_frequency               0
mean_purchase_gap_days      112080
median_purchase_gap_days    112080
max_purchase_gap_days       112080
purchase_gap_count          112080
observation_window_days          0
dtype: int64


Test missing values:


total_orders                    0
total_revenue                   0
average_order_value             0
recency_days                    0
customer_lifetime_days          0
repeat_customer                 0
purchase_frequency              0
mean_purchase_gap_days      54271
median_purchase_gap_days    54271
max_purchase_gap_days       54271
purchase_gap_count          54271
observation_window_days         0
dtype: int64

In [12]:
xgb_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "smote",
            SMOTE(
                random_state=42
            )
        ),

        (
            "model",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1,
                tree_method="hist"
            )
        )
    ]
)

print("XGBoost pipeline created.")

XGBoost pipeline created.


In [13]:
xgb_params = {

    "smote__sampling_strategy": [
        0.05,
        0.10,
        0.20,
        0.30
    ],

    "smote__k_neighbors": [
        3,
        5
    ],

    "model__n_estimators": [
        100,
        200,
        300
    ],

    "model__max_depth": [
        3,
        5,
        6
    ],

    "model__learning_rate": [
        0.03,
        0.05,
        0.10
    ],

    "model__subsample": [
        0.8,
        1.0
    ],

    "model__colsample_bytree": [
        0.8,
        1.0
    ],

    "model__min_child_weight": [
        3,
        5,
        10
    ]
}

print("XGBoost search space created.")

XGBoost search space created.


In [14]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

print(
    "3-Fold Stratified CV configured."
)

3-Fold Stratified CV configured.


In [15]:
xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,

    param_distributions=xgb_params,

    n_iter=12,

    scoring="recall",

    cv=cv,

    random_state=42,

    n_jobs=-1,

    verbose=1,

    return_train_score=True
)

print("RandomizedSearchCV configured.")

print("\nCandidates: 12")
print("CV folds: 3")
print("Total fits: 36")
print("Optimization metric: Recall")

RandomizedSearchCV configured.

Candidates: 12
CV folds: 3
Total fits: 36
Optimization metric: Recall


In [16]:
print("=" * 70)
print("TRAINING XGBOOST")
print("=" * 70)

xgb_search.fit(
    X_train,
    y_train
)

print("\nXGBoost training completed.")

TRAINING XGBOOST
Fitting 3 folds for each of 12 candidates, totalling 36 fits



XGBoost training completed.


In [17]:
print("=" * 70)
print("BEST XGBOOST CV RESULT")
print("=" * 70)

print(
    f"\nBest CV Recall: "
    f"{xgb_search.best_score_:.4f}"
)

print("\nBest Parameters:")

for parameter, value in xgb_search.best_params_.items():
    print(
        f"{parameter}: {value}"
    )

BEST XGBOOST CV RESULT

Best CV Recall: 0.0395

Best Parameters:
smote__sampling_strategy: 0.3
smote__k_neighbors: 5
model__subsample: 0.8
model__n_estimators: 300
model__min_child_weight: 5
model__max_depth: 5
model__learning_rate: 0.1
model__colsample_bytree: 0.8


In [18]:
xgb_model = (
    xgb_search.best_estimator_
)

print(
    "Best XGBoost model obtained."
)

Best XGBoost model obtained.


In [19]:
xgb_pred = xgb_model.predict(
    X_test
)

xgb_prob = (
    xgb_model
    .predict_proba(X_test)[:, 1]
)

print("Test predictions generated.")

print("\nPredicted classes:")

print(
    pd.Series(xgb_pred)
    .value_counts()
)

Test predictions generated.

Predicted classes:
0    55346
1      561
Name: count, dtype: int64


In [20]:
xgb_accuracy = accuracy_score(
    y_test,
    xgb_pred
)

xgb_precision = precision_score(
    y_test,
    xgb_pred,
    zero_division=0
)

xgb_recall = recall_score(
    y_test,
    xgb_pred,
    zero_division=0
)

xgb_f1 = f1_score(
    y_test,
    xgb_pred,
    zero_division=0
)

xgb_roc_auc = roc_auc_score(
    y_test,
    xgb_prob
)

xgb_pr_auc = average_precision_score(
    y_test,
    xgb_prob
)

print("=" * 70)
print("XGBOOST TEST RESULTS")
print("=" * 70)

print(
    f"\nAccuracy  : {xgb_accuracy:.4f}"
)

print(
    f"Precision : {xgb_precision:.4f}"
)

print(
    f"Recall    : {xgb_recall:.4f}"
)

print(
    f"F1 Score  : {xgb_f1:.4f}"
)

print(
    f"ROC-AUC   : {xgb_roc_auc:.4f}"
)

print(
    f"PR-AUC    : {xgb_pr_auc:.4f}"
)

XGBOOST TEST RESULTS

Accuracy  : 0.9795
Precision : 0.0642
Recall    : 0.0550
F1 Score  : 0.0592
ROC-AUC   : 0.5582
PR-AUC    : 0.0263


In [21]:
xgb_cm = confusion_matrix(
    y_test,
    xgb_pred
)

print(
    "XGBOOST CONFUSION MATRIX"
)

print("=" * 70)

print(
    xgb_cm
)

XGBOOST CONFUSION MATRIX
[[54727   525]
 [  619    36]]


In [22]:
print(
    classification_report(
        y_test,
        xgb_pred,
        digits=4,
        zero_division=0
    )
)

              precision    recall  f1-score   support

           0     0.9888    0.9905    0.9897     55252
           1     0.0642    0.0550    0.0592       655

    accuracy                         0.9795     55907
   macro avg     0.5265    0.5227    0.5244     55907
weighted avg     0.9780    0.9795    0.9788     55907



# New code

In [23]:
# ============================================================
# XGBOOST V2 - CLASS WEIGHTING WITHOUT SMOTE
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

print("=" * 70)
print("XGBOOST V2 - NO SMOTE")
print("=" * 70)

xgb_v2_pipeline = Pipeline(
    steps=[

        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "model",
            XGBClassifier(
                objective="binary:logistic",

                eval_metric="aucpr",

                tree_method="hist",

                random_state=42,

                n_jobs=2,

                verbosity=0
            )
        )
    ]
)

print("Pipeline created.")

XGBOOST V2 - NO SMOTE
Pipeline created.


In [24]:
# ============================================================
# XGBOOST V2 SEARCH SPACE
# ============================================================

xgb_v2_params = {

    "model__n_estimators": [
        150,
        250,
        400,
        600
    ],

    "model__max_depth": [
        2,
        3,
        4
    ],

    "model__learning_rate": [
        0.02,
        0.03,
        0.05
    ],

    "model__min_child_weight": [
        1,
        3,
        5,
        10
    ],

    "model__gamma": [
        0,
        0.1,
        0.25,
        0.5
    ],

    "model__subsample": [
        0.7,
        0.8,
        1.0
    ],

    "model__colsample_bytree": [
        0.7,
        0.8,
        1.0
    ],

    "model__reg_lambda": [
        1,
        5,
        10,
        20
    ],

    "model__reg_alpha": [
        0,
        0.1,
        0.5
    ],

    "model__scale_pos_weight": [
        1,
        2,
        5,
        10,
        20,
        40,
        60
    ]
}

print("Search space created.")

Search space created.


In [25]:
# ============================================================
# XGBOOST V2 - F2 SEARCH
# ============================================================

from sklearn.metrics import make_scorer, fbeta_score

f2_scorer = make_scorer(
    fbeta_score,
    beta=2,
    zero_division=0
)

xgb_v2_search = RandomizedSearchCV(

    estimator=xgb_v2_pipeline,

    param_distributions=xgb_v2_params,

    n_iter=40,

    scoring=f2_scorer,

    cv=cv,

    random_state=42,

    n_jobs=2,

    verbose=2,

    return_train_score=True
)

print("=" * 70)
print("XGBOOST V2 SEARCH")
print("=" * 70)

print("Candidates : 40")
print("CV folds   : 3")
print("Total fits : 120")
print("Metric     : F2")

XGBOOST V2 SEARCH
Candidates : 40
CV folds   : 3
Total fits : 120
Metric     : F2


In [26]:
# ============================================================
# TRAIN XGBOOST V2
# ============================================================

print("=" * 70)
print("STARTING XGBOOST V2 TRAINING")
print("=" * 70)

xgb_v2_search.fit(
    X_train,
    y_train
)

print("\nTraining completed.")

STARTING XGBOOST V2 TRAINING
Fitting 3 folds for each of 40 candidates, totalling 120 fits

Training completed.


In [27]:
# ============================================================
# BEST XGBOOST V2
# ============================================================

print("=" * 70)
print("BEST XGBOOST V2")
print("=" * 70)

print(
    "Best CV F2:",
    round(
        xgb_v2_search.best_score_,
        6
    )
)

print("\nBest parameters:")

for parameter, value in (
    xgb_v2_search.best_params_.items()
):

    print(
        f"{parameter}: {value}"
    )

xgb_v2_model = (
    xgb_v2_search.best_estimator_
)

print("\nBest model obtained.")

BEST XGBOOST V2
Best CV F2: 0.088601

Best parameters:
model__subsample: 0.7
model__scale_pos_weight: 60
model__reg_lambda: 10
model__reg_alpha: 0.5
model__n_estimators: 250
model__min_child_weight: 1
model__max_depth: 4
model__learning_rate: 0.05
model__gamma: 0.1
model__colsample_bytree: 0.8

Best model obtained.


In [28]:
# ============================================================
# XGBOOST V2 TEST PROBABILITY
# ============================================================

xgb_v2_prob = (
    xgb_v2_model
    .predict_proba(X_test)[:, 1]
)

print("=" * 70)
print("XGBOOST V2 PROBABILITY DISTRIBUTION")
print("=" * 70)

print(
    pd.Series(
        xgb_v2_prob
    ).describe()
)

XGBOOST V2 PROBABILITY DISTRIBUTION
count    55907.000000
mean         0.403092
std          0.119629
min          0.015351
25%          0.336100
50%          0.415753
75%          0.477782
max          0.952751
dtype: float64


In [29]:
# ============================================================
# THRESHOLD ANALYSIS
# ============================================================

threshold_results = []

for threshold in np.arange(
    0.001,
    0.501,
    0.001
):

    prediction = (
        xgb_v2_prob >= threshold
    ).astype(int)

    accuracy = accuracy_score(
        y_test,
        prediction
    )

    precision = precision_score(
        y_test,
        prediction,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        prediction,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        prediction,
        zero_division=0
    )

    f2 = fbeta_score(
        y_test,
        prediction,
        beta=2,
        zero_division=0
    )

    threshold_results.append({

        "threshold": threshold,

        "accuracy": accuracy,

        "precision": precision,

        "recall": recall,

        "f1": f1,

        "f2": f2,

        "predicted_positive":
            prediction.sum()
    })


threshold_df = pd.DataFrame(
    threshold_results
)

print("=" * 70)
print("BEST THRESHOLDS BY F2")
print("=" * 70)

display(
    threshold_df
    .sort_values(
        "f2",
        ascending=False
    )
    .head(20)
)

BEST THRESHOLDS BY F2


,threshold,accuracy,precision,recall,f1,f2,predicted_positive
493,0.494,0.794015,0.018359,0.316031,0.034702,0.074487,11275
495,0.496,0.800705,0.018371,0.305344,0.034656,0.074036,10887
492,0.493,0.791100,0.018182,0.317557,0.034394,0.073969,11440
494,0.495,0.796805,0.018270,0.309924,0.034506,0.073920,11111
496,0.497,0.803817,0.018399,0.300763,0.034677,0.073910,10707
497,0.498,0.805802,0.018319,0.296183,0.034504,0.073429,10590
491,0.492,0.787075,0.017914,0.319084,0.033923,0.073143,11667
490,0.491,0.783354,0.017844,0.323664,0.033823,0.073098,11881
498,0.499,0.808825,0.018246,0.290076,0.034333,0.072892,10413
489,0.490,0.780350,0.017675,0.325191,0.033527,0.072592,12051


In [30]:
# ============================================================
# CREATE VALIDATION SET FROM TRAINING DATA
# ============================================================

from sklearn.model_selection import train_test_split

X_fit, X_valid, y_fit, y_valid = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    stratify=y_train,
    random_state=42
)

print("=" * 70)
print("VALIDATION SPLIT")
print("=" * 70)

print("X_fit   :", X_fit.shape)
print("X_valid :", X_valid.shape)

print(
    "\ny_fit positive rate:",
    f"{y_fit.mean() * 100:.4f}%"
)

print(
    "y_valid positive rate:",
    f"{y_valid.mean() * 100:.4f}%"
)

VALIDATION SPLIT
X_fit   : (92050, 12)
X_valid : (23013, 12)

y_fit positive rate: 1.4514%
y_valid positive rate: 1.4514%


In [31]:
# ============================================================
# FIT BEST XGBOOST CONFIGURATION ON FIT DATA
# ============================================================

best_params = {
    key.replace(
        "model__",
        ""
    ): value

    for key, value
    in xgb_v2_search.best_params_.items()
}

validation_model = Pipeline(
    steps=[

        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "model",
            XGBClassifier(
                objective="binary:logistic",

                eval_metric="aucpr",

                tree_method="hist",

                random_state=42,

                n_jobs=2,

                verbosity=0,

                **best_params
            )
        )
    ]
)

print("=" * 70)
print("TRAINING VALIDATION MODEL")
print("=" * 70)

validation_model.fit(
    X_fit,
    y_fit
)

print("Validation model trained.")

TRAINING VALIDATION MODEL
Validation model trained.


In [32]:
# ============================================================
# SELECT THRESHOLD USING VALIDATION ONLY
# ============================================================

valid_probability = (
    validation_model
    .predict_proba(X_valid)[:, 1]
)

validation_thresholds = []

for threshold in np.arange(
    0.001,
    0.501,
    0.001
):

    prediction = (
        valid_probability >= threshold
    ).astype(int)

    precision = precision_score(
        y_valid,
        prediction,
        zero_division=0
    )

    recall = recall_score(
        y_valid,
        prediction,
        zero_division=0
    )

    f1 = f1_score(
        y_valid,
        prediction,
        zero_division=0
    )

    f2 = fbeta_score(
        y_valid,
        prediction,
        beta=2,
        zero_division=0
    )

    validation_thresholds.append({

        "threshold": threshold,

        "precision": precision,

        "recall": recall,

        "f1": f1,

        "f2": f2
    })


validation_threshold_df = pd.DataFrame(
    validation_thresholds
)

print("=" * 70)
print("VALIDATION THRESHOLDS")
print("=" * 70)

display(
    validation_threshold_df
    .sort_values(
        "f2",
        ascending=False
    )
    .head(20)
)

VALIDATION THRESHOLDS


,threshold,precision,recall,f1,f2
457,0.458,0.020598,0.502994,0.039576,0.088496
456,0.457,0.020366,0.505988,0.039157,0.087710
453,0.454,0.020211,0.520958,0.038913,0.087481
458,0.459,0.020347,0.491018,0.039076,0.087271
452,0.453,0.020099,0.523952,0.038713,0.087125
455,0.456,0.020165,0.505988,0.038784,0.086961
442,0.443,0.019752,0.577844,0.038199,0.086882
454,0.455,0.020045,0.511976,0.038579,0.086652
459,0.460,0.020213,0.482036,0.038800,0.086550
449,0.450,0.019839,0.538922,0.038269,0.086464


In [33]:
# ============================================================
# SELECT BEST VALIDATION THRESHOLD
# ============================================================

best_threshold_row = (
    validation_threshold_df
    .sort_values(
        ["f2", "precision"],
        ascending=False
    )
    .iloc[0]
)

best_threshold_v2 = float(
    best_threshold_row["threshold"]
)

print("=" * 70)
print("SELECTED THRESHOLD")
print("=" * 70)

print(
    f"Threshold : {best_threshold_v2:.4f}"
)

print(
    f"Precision : "
    f"{best_threshold_row['precision']:.4f}"
)

print(
    f"Recall    : "
    f"{best_threshold_row['recall']:.4f}"
)

print(
    f"F1        : "
    f"{best_threshold_row['f1']:.4f}"
)

print(
    f"F2        : "
    f"{best_threshold_row['f2']:.4f}"
)

SELECTED THRESHOLD
Threshold : 0.4580
Precision : 0.0206
Recall    : 0.5030
F1        : 0.0396
F2        : 0.0885


In [34]:
# ============================================================
# FINAL UNTOUCHED TEST
# ============================================================

final_test_probability = (
    validation_model
    .predict_proba(X_test)[:, 1]
)

final_test_prediction = (
    final_test_probability
    >= best_threshold_v2
).astype(int)


final_accuracy = accuracy_score(
    y_test,
    final_test_prediction
)

final_precision = precision_score(
    y_test,
    final_test_prediction,
    zero_division=0
)

final_recall = recall_score(
    y_test,
    final_test_prediction,
    zero_division=0
)

final_f1 = f1_score(
    y_test,
    final_test_prediction,
    zero_division=0
)

final_f2 = fbeta_score(
    y_test,
    final_test_prediction,
    beta=2,
    zero_division=0
)

final_roc_auc = roc_auc_score(
    y_test,
    final_test_probability
)

final_pr_auc = average_precision_score(
    y_test,
    final_test_probability
)


print("=" * 70)
print("XGBOOST V2 - FINAL TEST")
print("=" * 70)

print(
    f"Threshold  : {best_threshold_v2:.4f}"
)

print(
    f"Accuracy   : {final_accuracy:.4f}"
)

print(
    f"Precision  : {final_precision:.4f}"
)

print(
    f"Recall     : {final_recall:.4f}"
)

print(
    f"F1 Score   : {final_f1:.4f}"
)

print(
    f"F2 Score   : {final_f2:.4f}"
)

print(
    f"ROC-AUC    : {final_roc_auc:.4f}"
)

print(
    f"PR-AUC     : {final_pr_auc:.4f}"
)

XGBOOST V2 - FINAL TEST
Threshold  : 0.4580
Accuracy   : 0.6621
Precision  : 0.0150
Recall     : 0.4305
F1 Score   : 0.0290
F2 Score   : 0.0658
ROC-AUC    : 0.5881
PR-AUC     : 0.0257


In [35]:
# ============================================================
# MODEL COMPARISON
# ============================================================

comparison = pd.DataFrame({

    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "PR-AUC"
    ],

    "XGBoost V1": [
        xgb_accuracy,
        xgb_precision,
        xgb_recall,
        xgb_f1,
        xgb_roc_auc,
        xgb_pr_auc
    ],

    "XGBoost V2": [
        final_accuracy,
        final_precision,
        final_recall,
        final_f1,
        final_roc_auc,
        final_pr_auc
    ]
})

display(comparison)

,Metric,XGBoost V1,XGBoost V2
0,Accuracy,0.979537,0.662064
1,Precision,0.064171,0.014998
2,Recall,0.054962,0.430534
3,F1,0.059211,0.028987
4,ROC-AUC,0.558164,0.588122
5,PR-AUC,0.026323,0.025707


In [41]:
# Check currently available XGBoost/model variables

[x for x in globals().keys() if "xgb" in x.lower() or "model" in x.lower()]

['XGBClassifier',
 'xgb_v2_pipeline',
 'MODEL_DIR',
 'xgb_pipeline',
 'xgb_params',
 'xgb_search',
 'xgb_model',
 'xgb_pred',
 'xgb_prob',
 'xgb_accuracy',
 'xgb_precision',
 'xgb_recall',
 'xgb_f1',
 'xgb_roc_auc',
 'xgb_pr_auc',
 'xgb_cm',
 'xgb_v2_params',
 'xgb_v2_search',
 'xgb_v2_model',
 'xgb_v2_prob',
 'validation_model']

In [42]:
# ============================================================
# V2 XGBOOST - PREDICTIONS
# ============================================================

v2_pred = xgb_v2_model.predict(X_test)

print("V2 predictions created successfully.")
print("Total predictions:", len(v2_pred))

V2 predictions created successfully.
Total predictions: 55907


In [43]:
# ============================================================
# V2 XGBOOST - CONFUSION MATRIX
# ============================================================

from sklearn.metrics import confusion_matrix

cm_v2 = confusion_matrix(y_test, v2_pred)

print("V2 Confusion Matrix:")
print(cm_v2)

tn, fp, fn, tp = cm_v2.ravel()

print("\nDetailed Results:")
print(f"True Negatives  (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives  (TP): {tp}")

print(f"\nRecall:    {tp / (tp + fn):.4f}")
print(f"Precision: {tp / (tp + fp):.4f}")

V2 Confusion Matrix:
[[45198 10054]
 [  469   186]]

Detailed Results:
True Negatives  (TN): 45198
False Positives (FP): 10054
False Negatives (FN): 469
True Positives  (TP): 186

Recall:    0.2840
Precision: 0.0182


In [44]:
# ============================================================
# V2 XGBOOST - CLASSIFICATION REPORT
# ============================================================

from sklearn.metrics import classification_report

v2_pred = xgb_v2_model.predict(X_test)

print("V2 XGBoost Classification Report:\n")

print(
    classification_report(
        y_test,
        v2_pred,
        target_names=["No Repeat", "Repeat"],
        digits=4,
        zero_division=0
    )
)

V2 XGBoost Classification Report:

              precision    recall  f1-score   support

   No Repeat     0.9897    0.8180    0.8957     55252
      Repeat     0.0182    0.2840    0.0341       655

    accuracy                         0.8118     55907
   macro avg     0.5039    0.5510    0.4649     55907
weighted avg     0.9783    0.8118    0.8856     55907



In [45]:
# ============================================================
# V3 - PRODUCTION CANDIDATE
# Feature Engineering + SMOTE + XGBoost
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier


# ------------------------------------------------------------
# 1. COPY DATA
# ------------------------------------------------------------

train_v3 = train_df.copy()
test_v3 = test_df.copy()


# ------------------------------------------------------------
# 2. FEATURE ENGINEERING
# Only information available at prediction time
# ------------------------------------------------------------

for df in [train_v3, test_v3]:

    # Revenue per order
    df["revenue_per_order"] = (
        df["total_revenue"] /
        df["total_orders"].replace(0, np.nan)
    )

    # Orders per lifetime day
    df["orders_per_lifetime_day"] = (
        df["total_orders"] /
        df["customer_lifetime_days"].replace(0, np.nan)
    )

    # Revenue per lifetime day
    df["revenue_per_lifetime_day"] = (
        df["total_revenue"] /
        df["customer_lifetime_days"].replace(0, np.nan)
    )

    # Recency relative to observation window
    df["recency_ratio"] = (
        df["recency_days"] /
        df["observation_window_days"].replace(0, np.nan)
    )

    # Purchase frequency relative to observation period
    df["purchase_frequency_ratio"] = (
        df["purchase_frequency"] /
        df["observation_window_days"].replace(0, np.nan)
    )

    # Average gap relative to observation period
    df["gap_ratio"] = (
        df["mean_purchase_gap_days"] /
        df["observation_window_days"].replace(0, np.nan)
    )

    # Revenue intensity
    df["revenue_order_frequency"] = (
        df["total_revenue"] /
        (df["total_orders"] + 1)
    )


# ------------------------------------------------------------
# 3. TARGET
# ------------------------------------------------------------

target = "repeat_purchase"

y_train_v3 = train_v3[target].astype(int)
y_test_v3 = test_v3[target].astype(int)


# ------------------------------------------------------------
# 4. REMOVE ID / DATE / FUTURE / TARGET COLUMNS
# ------------------------------------------------------------

drop_columns = [
    "customer_unique_id",
    "snapshot_date",
    "future_end_date",
    "first_purchase_date",
    "last_purchase_date",
    "future_purchase_count",
    "churn_label",
    "repeat_purchase"
]

X_train_v3 = train_v3.drop(
    columns=drop_columns,
    errors="ignore"
)

X_test_v3 = test_v3.drop(
    columns=drop_columns,
    errors="ignore"
)


# ------------------------------------------------------------
# 5. MAKE SURE BOTH DATASETS HAVE SAME COLUMNS
# ------------------------------------------------------------

X_test_v3 = X_test_v3[X_train_v3.columns]


# ------------------------------------------------------------
# 6. IMPUTATION
# ------------------------------------------------------------

imputer_v3 = SimpleImputer(strategy="median")

X_train_v3 = pd.DataFrame(
    imputer_v3.fit_transform(X_train_v3),
    columns=X_train_v3.columns,
    index=X_train_v3.index
)

X_test_v3 = pd.DataFrame(
    imputer_v3.transform(X_test_v3),
    columns=X_test_v3.columns,
    index=X_test_v3.index
)


# ------------------------------------------------------------
# 7. CHECK ORIGINAL IMBALANCE
# ------------------------------------------------------------

print("BEFORE SMOTE")
print(y_train_v3.value_counts())
print("\nOriginal positive rate:",
      y_train_v3.mean())


# ------------------------------------------------------------
# 8. SMOTE - TRAINING DATA ONLY
# ------------------------------------------------------------

smote_v3 = SMOTE(
    sampling_strategy=0.20,
    random_state=42,
    k_neighbors=5
)

X_train_balanced_v3, y_train_balanced_v3 = smote_v3.fit_resample(
    X_train_v3,
    y_train_v3
)


print("\nAFTER SMOTE")
print(pd.Series(y_train_balanced_v3).value_counts())

print(
    "\nBalanced positive rate:",
    y_train_balanced_v3.mean()
)


# ------------------------------------------------------------
# 9. FINAL XGBOOST MODEL
# ------------------------------------------------------------

xgb_v3_model = XGBClassifier(
    objective="binary:logistic",

    n_estimators=300,
    max_depth=3,
    learning_rate=0.03,

    min_child_weight=5,
    subsample=0.85,
    colsample_bytree=0.85,

    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=5,

    eval_metric="aucpr",

    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)


# ------------------------------------------------------------
# 10. TRAIN
# ------------------------------------------------------------

xgb_v3_model.fit(
    X_train_balanced_v3,
    y_train_balanced_v3
)


# ------------------------------------------------------------
# 11. TEST PREDICTIONS
# ------------------------------------------------------------

v3_prob = xgb_v3_model.predict_proba(
    X_test_v3
)[:, 1]

v3_pred = (v3_prob >= 0.50).astype(int)


# ------------------------------------------------------------
# 12. METRICS
# ------------------------------------------------------------

v3_accuracy = accuracy_score(
    y_test_v3,
    v3_pred
)

v3_balanced_accuracy = balanced_accuracy_score(
    y_test_v3,
    v3_pred
)

v3_precision = precision_score(
    y_test_v3,
    v3_pred,
    zero_division=0
)

v3_recall = recall_score(
    y_test_v3,
    v3_pred,
    zero_division=0
)

v3_f1 = f1_score(
    y_test_v3,
    v3_pred,
    zero_division=0
)

v3_roc_auc = roc_auc_score(
    y_test_v3,
    v3_prob
)

v3_pr_auc = average_precision_score(
    y_test_v3,
    v3_prob
)


# ------------------------------------------------------------
# 13. FINAL RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("V3 XGBOOST - FINAL RESULTS")
print("=" * 60)

print(f"Accuracy:           {v3_accuracy:.4f}")
print(f"Balanced Accuracy:  {v3_balanced_accuracy:.4f}")
print(f"Precision:          {v3_precision:.4f}")
print(f"Recall:             {v3_recall:.4f}")
print(f"F1 Score:           {v3_f1:.4f}")
print(f"ROC-AUC:            {v3_roc_auc:.4f}")
print(f"PR-AUC:             {v3_pr_auc:.4f}")


# ------------------------------------------------------------
# 14. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nClassification Report:\n")

print(
    classification_report(
        y_test_v3,
        v3_pred,
        target_names=[
            "No Repeat",
            "Repeat"
        ],
        digits=4,
        zero_division=0
    )
)


# ------------------------------------------------------------
# 15. CONFUSION MATRIX
# ------------------------------------------------------------

cm_v3 = confusion_matrix(
    y_test_v3,
    v3_pred
)

print("\nConfusion Matrix:")
print(cm_v3)

tn, fp, fn, tp = cm_v3.ravel()

print("\nTN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)


# ------------------------------------------------------------
# 16. PRODUCTION CHECK
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PRODUCTION CHECK")
print("=" * 60)

if v3_recall >= 0.70:
    print("Recall target: PASS")
else:
    print("Recall target: NOT MET")

if v3_precision >= 0.10:
    print("Precision >= 10%: PASS")
else:
    print("Precision >= 10%: NOT MET")

if v3_balanced_accuracy >= 0.70:
    print("Balanced Accuracy >= 70%: PASS")
else:
    print("Balanced Accuracy >= 70%: NOT MET")

if v3_pr_auc > 0.02:
    print("PR-AUC above baseline: CHECK")
else:
    print("PR-AUC: WEAK")

print("\nPositive class prevalence:",
      y_test_v3.mean())

BEFORE SMOTE
repeat_purchase
0    113393
1      1670
Name: count, dtype: int64

Original positive rate: 0.01451378809869376

AFTER SMOTE
repeat_purchase
0    113393
1     22678
Name: count, dtype: int64

Balanced positive rate: 0.16666299211441085

V3 XGBOOST - FINAL RESULTS
Accuracy:           0.9861
Balanced Accuracy:  0.5132
Precision:          0.1187
Recall:             0.0290
F1 Score:           0.0466
ROC-AUC:            0.5912
PR-AUC:             0.0338

Classification Report:

              precision    recall  f1-score   support

   No Repeat     0.9886    0.9974    0.9930     55252
      Repeat     0.1187    0.0290    0.0466       655

    accuracy                         0.9861     55907
   macro avg     0.5537    0.5132    0.5198     55907
weighted avg     0.9784    0.9861    0.9819     55907


Confusion Matrix:
[[55111   141]
 [  636    19]]

TN: 55111
FP: 141
FN: 636
TP: 19

PRODUCTION CHECK
Recall target: NOT MET
Precision >= 10%: PASS
Balanced Accuracy >= 70%: NOT MET
P

In [46]:
# ============================================================
# V3 XGBOOST - CLASSIFICATION REPORT
# ============================================================

from sklearn.metrics import classification_report

print("V3 XGBoost Classification Report:\n")

print(
    classification_report(
        y_test_v3,
        v3_pred,
        target_names=["No Repeat", "Repeat"],
        digits=4,
        zero_division=0
    )
)

V3 XGBoost Classification Report:

              precision    recall  f1-score   support

   No Repeat     0.9886    0.9974    0.9930     55252
      Repeat     0.1187    0.0290    0.0466       655

    accuracy                         0.9861     55907
   macro avg     0.5537    0.5132    0.5198     55907
weighted avg     0.9784    0.9861    0.9819     55907



In [47]:
# ============================================================
# V4 XGBOOST - RANDOMIZED SEARCH + IMBALANCE HANDLING
# ============================================================

import numpy as np
import pandas as pd

from xgboost import XGBClassifier

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    make_scorer,
    fbeta_score
)


# ============================================================
# 1. F2 SCORER
# Recall ko precision se zyada importance
# ============================================================

f2_scorer = make_scorer(
    fbeta_score,
    beta=2,
    zero_division=0
)


# ============================================================
# 2. BASE XGBOOST
# ============================================================

xgb_v4_base = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)


# ============================================================
# 3. RANDOM SEARCH PARAMETERS
# ============================================================

xgb_v4_params = {
    "n_estimators": [100, 150, 200, 300, 400],
    "max_depth": [2, 3, 4, 5],
    "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08],
    "min_child_weight": [1, 3, 5, 8, 10],
    "gamma": [0, 0.05, 0.1, 0.2, 0.5],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "reg_alpha": [0, 0.01, 0.1, 0.5, 1],
    "reg_lambda": [1, 3, 5, 10, 20],

    # Severe class imbalance
    "scale_pos_weight": [
        10,
        20,
        30,
        40,
        50,
        60,
        70,
        80,
        100
    ]
}


# ============================================================
# 4. CROSS VALIDATION
# ============================================================

cv_v4 = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)


# ============================================================
# 5. RANDOMIZED SEARCH
# ============================================================

xgb_v4_search = RandomizedSearchCV(
    estimator=xgb_v4_base,
    param_distributions=xgb_v4_params,

    n_iter=30,

    scoring=f2_scorer,

    cv=cv_v4,

    n_jobs=-1,
    verbose=1,

    random_state=42,

    refit=True,
    return_train_score=True
)


# ============================================================
# 6. TRAIN
# ============================================================

xgb_v4_search.fit(
    X_train,
    y_train
)


# ============================================================
# 7. BEST PARAMETERS
# ============================================================

print("\n" + "=" * 70)
print("V4 BEST PARAMETERS")
print("=" * 70)

print(xgb_v4_search.best_params_)

print("\nBest CV F2:")
print(f"{xgb_v4_search.best_score_:.4f}")


# ============================================================
# 8. FINAL V4 MODEL
# ============================================================

xgb_v4_model = xgb_v4_search.best_estimator_


# ============================================================
# 9. TEST PROBABILITY
# ============================================================

v4_prob = xgb_v4_model.predict_proba(
    X_test
)[:, 1]


# ============================================================
# 10. DEFAULT THRESHOLD
# ============================================================

v4_pred = (
    v4_prob >= 0.50
).astype(int)


# ============================================================
# 11. METRICS
# ============================================================

v4_accuracy = accuracy_score(
    y_test,
    v4_pred
)

v4_balanced_accuracy = balanced_accuracy_score(
    y_test,
    v4_pred
)

v4_precision = precision_score(
    y_test,
    v4_pred,
    zero_division=0
)

v4_recall = recall_score(
    y_test,
    v4_pred,
    zero_division=0
)

v4_f1 = f1_score(
    y_test,
    v4_pred,
    zero_division=0
)

v4_f2 = fbeta_score(
    y_test,
    v4_pred,
    beta=2,
    zero_division=0
)

v4_roc_auc = roc_auc_score(
    y_test,
    v4_prob
)

v4_pr_auc = average_precision_score(
    y_test,
    v4_prob
)


# ============================================================
# 12. RESULTS
# ============================================================

print("\n" + "=" * 70)
print("V4 XGBOOST - TEST RESULTS")
print("=" * 70)

print(f"Accuracy:           {v4_accuracy:.4f}")
print(f"Balanced Accuracy:  {v4_balanced_accuracy:.4f}")
print(f"Precision:          {v4_precision:.4f}")
print(f"Recall:             {v4_recall:.4f}")
print(f"F1 Score:           {v4_f1:.4f}")
print(f"F2 Score:           {v4_f2:.4f}")
print(f"ROC-AUC:            {v4_roc_auc:.4f}")
print(f"PR-AUC:             {v4_pr_auc:.4f}")


# ============================================================
# 13. CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 70)
print("V4 CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_test,
        v4_pred,
        target_names=[
            "No Repeat",
            "Repeat"
        ],
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 14. CONFUSION MATRIX
# ============================================================

cm_v4 = confusion_matrix(
    y_test,
    v4_pred
)

print("\n" + "=" * 70)
print("V4 CONFUSION MATRIX")
print("=" * 70)

print(cm_v4)

tn, fp, fn, tp = cm_v4.ravel()

print("\nTN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

Fitting 3 folds for each of 30 candidates, totalling 90 fits

V4 BEST PARAMETERS
{'subsample': 0.7, 'scale_pos_weight': 50, 'reg_lambda': 3, 'reg_alpha': 0, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 2, 'learning_rate': 0.08, 'gamma': 0.1, 'colsample_bytree': 0.7}

Best CV F2:
0.0889

V4 XGBOOST - TEST RESULTS
Accuracy:           0.9016
Balanced Accuracy:  0.5504
Precision:          0.0245
Recall:             0.1908
F1 Score:           0.0435
F2 Score:           0.0810
ROC-AUC:            0.5966
PR-AUC:             0.0352

V4 CLASSIFICATION REPORT
              precision    recall  f1-score   support

   No Repeat     0.9896    0.9100    0.9481     55252
      Repeat     0.0245    0.1908    0.0435       655

    accuracy                         0.9016     55907
   macro avg     0.5070    0.5504    0.4958     55907
weighted avg     0.9783    0.9016    0.9375     55907


V4 CONFUSION MATRIX
[[50280  4972]
 [  530   125]]

TN: 50280
FP: 4972
FN: 530
TP: 125


In [48]:
# ============================================================
# V4 XGBOOST - CLASSIFICATION REPORT
# ============================================================

from sklearn.metrics import classification_report

print("V4 XGBoost Classification Report:\n")

print(
    classification_report(
        y_test,
        v4_pred,
        target_names=["No Repeat", "Repeat"],
        digits=4,
        zero_division=0
    )
)

V4 XGBoost Classification Report:

              precision    recall  f1-score   support

   No Repeat     0.9896    0.9100    0.9481     55252
      Repeat     0.0245    0.1908    0.0435       655

    accuracy                         0.9016     55907
   macro avg     0.5070    0.5504    0.4958     55907
weighted avg     0.9783    0.9016    0.9375     55907

